In [1]:
import sqlite3

# Connect to SQLite database
connection = sqlite3.connect("bus_pass_database.db")

cursor = connection.cursor()

# Create buses table
cursor.execute("""
CREATE TABLE IF NOT EXISTS buses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    bus_number TEXT UNIQUE NOT NULL,
    source TEXT NOT NULL,
    destination TEXT NOT NULL,
    price INTEGER NOT NULL,
    total_seats INTEGER NOT NULL
)
""")

connection.commit()

print("Bus database created successfully!")


Bus database created successfully!


In [2]:
# Add sample bus details

buses = [
    ("TS01AB1234", "Hyderabad", "Warangal", 250, 40),
    ("TS02CD5678", "Hyderabad", "Khammam", 300, 40),
    ("TS03EF9012", "Warangal", "Khammam", 180, 40),
    ("TS04GH3456", "Hyderabad", "Karimnagar", 280, 40)
]

cursor.executemany("""
INSERT OR IGNORE INTO buses
(bus_number, source, destination, price, total_seats)
VALUES (?, ?, ?, ?, ?)
""", buses)

connection.commit()

print("Bus details added successfully!")

Bus details added successfully!


In [3]:
import sqlite3
import uuid
import pandas as pd
from datetime import datetime, date

In [4]:
# Connect to SQLite database
connection = sqlite3.connect("bus_pass_database.db")

cursor = connection.cursor()

print("Database connected successfully!")

Database connected successfully!


In [5]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS buses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    bus_number TEXT UNIQUE NOT NULL,
    source TEXT NOT NULL,
    destination TEXT NOT NULL,
    price INTEGER NOT NULL,
    total_seats INTEGER NOT NULL
)
""")

connection.commit()

print("Buses table created successfully!")

Buses table created successfully!


In [6]:
buses = [
    ("TS01AB1234", "Hyderabad", "Warangal", 250, 40),
    ("TS02CD5678", "Hyderabad", "Khammam", 300, 40),
    ("TS03EF9012", "Warangal", "Khammam", 180, 40),
    ("TS04GH3456", "Hyderabad", "Karimnagar", 280, 40)
]

cursor.executemany("""
INSERT OR IGNORE INTO buses
(bus_number, source, destination, price, total_seats)
VALUES (?, ?, ?, ?, ?)
""", buses)

connection.commit()

print("Bus details added successfully!")

Bus details added successfully!


In [7]:
bus_data = pd.read_sql_query(
    "SELECT * FROM buses",
    connection
)

bus_data

,id,bus_number,source,destination,price,total_seats
0,1,TS01AB1234,Hyderabad,Warangal,250,40
1,2,TS02CD5678,Hyderabad,Khammam,300,40
2,3,TS03EF9012,Warangal,Khammam,180,40
3,4,TS04GH3456,Hyderabad,Karimnagar,280,40


In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS bookings (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    ticket_id TEXT UNIQUE NOT NULL,
    passenger_name TEXT NOT NULL,
    bus_id INTEGER NOT NULL,
    journey_date TEXT NOT NULL,
    seat_number INTEGER NOT NULL,
    price INTEGER NOT NULL,
    booking_time TEXT NOT NULL,
    UNIQUE(bus_id, journey_date, seat_number)
)
""")

connection.commit()

print("Bookings table created successfully!")

Bookings table created successfully!


In [9]:
def search_buses(source, destination):
    
    query = """
    SELECT *
    FROM buses
    WHERE LOWER(source) = LOWER(?)
    AND LOWER(destination) = LOWER(?)
    """
    
    result = pd.read_sql_query(
        query,
        connection,
        params=(source, destination)
    )
    
    return result

In [10]:
search_buses("Hyderabad", "Warangal")

,id,bus_number,source,destination,price,total_seats
0,1,TS01AB1234,Hyderabad,Warangal,250,40


In [11]:
def get_available_seats(bus_id, journey_date):
    
    bus = connection.execute("""
        SELECT total_seats
        FROM buses
        WHERE id = ?
    """, (bus_id,)).fetchone()
    
    if bus is None:
        return []
    
    total_seats = bus[0]
    
    booked = connection.execute("""
        SELECT seat_number
        FROM bookings
        WHERE bus_id = ?
        AND journey_date = ?
    """, (bus_id, journey_date)).fetchall()
    
    booked_seats = {row[0] for row in booked}
    
    available = [
        seat
        for seat in range(1, total_seats + 1)
        if seat not in booked_seats
    ]
    
    return available

In [12]:
test_date = "2026-10-01"

available_seats = get_available_seats(1, test_date)

print("Available seats:")
print(available_seats)

Available seats:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]


In [13]:
def book_ticket(passenger_name, bus_id, journey_date):
    
    passenger_name = passenger_name.strip()
    
    if not passenger_name:
        return "Booking failed: Passenger name is required."
    
    # Validate date
    try:
        selected_date = datetime.strptime(
            journey_date,
            "%Y-%m-%d"
        ).date()
    except ValueError:
        return "Booking failed: Invalid date."
    
    if selected_date < date.today():
        return "Booking failed: Journey date cannot be in the past."
    
    # Get bus information
    bus = connection.execute("""
        SELECT bus_number, source, destination, price, total_seats
        FROM buses
        WHERE id = ?
    """, (bus_id,)).fetchone()
    
    if bus is None:
        return "Booking failed: Bus not found."
    
    bus_number, source, destination, price, total_seats = bus
    
    # Find available seats
    available_seats = get_available_seats(
        bus_id,
        journey_date
    )
    
    if not available_seats:
        return "Booking failed: No seats available."
    
    # Automatically assign first available seat
    seat_number = available_seats[0]
    
    # Generate unique ticket ID
    ticket_id = "CA-" + uuid.uuid4().hex[:10].upper()
    
    booking_time = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )
    
    try:
        
        cursor.execute("""
            INSERT INTO bookings
            (
                ticket_id,
                passenger_name,
                bus_id,
                journey_date,
                seat_number,
                price,
                booking_time
            )
            VALUES (?, ?, ?, ?, ?, ?, ?)
        """, (
            ticket_id,
            passenger_name,
            bus_id,
            journey_date,
            seat_number,
            price,
            booking_time
        ))
        
        connection.commit()
        
        return {
            "ticket_id": ticket_id,
            "passenger_name": passenger_name,
            "bus_number": bus_number,
            "source": source,
            "destination": destination,
            "journey_date": journey_date,
            "seat_number": seat_number,
            "price": price,
            "booking_time": booking_time
        }
        
    except sqlite3.IntegrityError:
        
        connection.rollback()
        
        return "Booking failed: Seat is already booked."

In [14]:
booking1 = book_ticket(
    "Rahul Kumar",
    1,
    "2026-10-01"
)

booking1

{'ticket_id': 'CA-ACEF36238A',
 'passenger_name': 'Rahul Kumar',
 'bus_number': 'TS01AB1234',
 'source': 'Hyderabad',
 'destination': 'Warangal',
 'journey_date': '2026-10-01',
 'seat_number': 1,
 'price': 250,
 'booking_time': '2026-09-13 16:20:45'}

In [15]:
booking2 = book_ticket(
    "Anu Sharma",
    1,
    "2026-10-01"
)

booking2

{'ticket_id': 'CA-639EF19F3B',
 'passenger_name': 'Anu Sharma',
 'bus_number': 'TS01AB1234',
 'source': 'Hyderabad',
 'destination': 'Warangal',
 'journey_date': '2026-10-01',
 'seat_number': 2,
 'price': 250,
 'booking_time': '2026-09-13 16:23:29'}

In [16]:
all_bookings = pd.read_sql_query("""
SELECT *
FROM bookings
""", connection)

all_bookings

,id,ticket_id,passenger_name,bus_id,journey_date,seat_number,price,booking_time
0,1,CA-ACEF36238A,Rahul Kumar,1,2026-10-01,1,250,2026-09-13 16:20:45
1,2,CA-639EF19F3B,Anu Sharma,1,2026-10-01,2,250,2026-09-13 16:23:29


In [17]:
available_seats = get_available_seats(
    1,
    "2026-10-01"
)

print("Available seats:")
print(available_seats)

Available seats:
[3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40]


In [18]:
def verify_ticket(ticket_id):
    
    query = """
    SELECT
        b.ticket_id,
        b.passenger_name,
        bus.bus_number,
        bus.source,
        bus.destination,
        b.journey_date,
        b.seat_number,
        b.price,
        b.booking_time
    FROM bookings b
    JOIN buses bus
    ON b.bus_id = bus.id
    WHERE b.ticket_id = ?
    """
    
    result = connection.execute(
        query,
        (ticket_id,)
    ).fetchone()
    
    if result:
        return result
    
    return "Ticket not found."

In [19]:
verify_ticket(booking1["ticket_id"])

('CA-ACEF36238A',
 'Rahul Kumar',
 'TS01AB1234',
 'Hyderabad',
 'Warangal',
 '2026-10-01',
 1,
 250,
 '2026-09-13 16:20:45')

In [20]:
verify_ticket("CA-INVALID123")

'Ticket not found.'

In [21]:
cursor.execute("""
SELECT bus_id, journey_date, seat_number, COUNT(*)
FROM bookings
GROUP BY bus_id, journey_date, seat_number
HAVING COUNT(*) > 1
""")

duplicates = cursor.fetchall()

if len(duplicates) == 0:
    print("SUCCESS: No duplicate seat bookings found.")
else:
    print("Duplicate bookings detected:")
    print(duplicates)

SUCCESS: No duplicate seat bookings found.


In [22]:
price_check = pd.read_sql_query("""
SELECT
    b.ticket_id,
    bus.bus_number,
    bus.source,
    bus.destination,
    b.price
FROM bookings b
JOIN buses bus
ON b.bus_id = bus.id
""", connection)

price_check

,ticket_id,bus_number,source,destination,price
0,CA-ACEF36238A,TS01AB1234,Hyderabad,Warangal,250
1,CA-639EF19F3B,TS01AB1234,Hyderabad,Warangal,250


In [23]:
import time

start_time = time.time()

search_results = []

for i in range(100):
    
    result = search_buses(
        "Hyderabad",
        "Warangal"
    )
    
    search_results.append(len(result))

end_time = time.time()

print("Requests simulated:", 100)
print("Successful searches:", sum(x > 0 for x in search_results))
print("Execution time:", round(end_time - start_time, 4), "seconds")

Requests simulated: 100
Successful searches: 100
Execution time: 0.18 seconds


In [24]:
print("========== SYSTEM RELIABILITY CHECK ==========")

# Check buses
bus_count = connection.execute(
    "SELECT COUNT(*) FROM buses"
).fetchone()[0]

# Check bookings
booking_count = connection.execute(
    "SELECT COUNT(*) FROM bookings"
).fetchone()[0]

# Check duplicate bookings
duplicate_count = connection.execute("""
SELECT COUNT(*)
FROM (
    SELECT bus_id, journey_date, seat_number
    FROM bookings
    GROUP BY bus_id, journey_date, seat_number
    HAVING COUNT(*) > 1
)
""").fetchone()[0]

print("Total buses:", bus_count)
print("Total bookings:", booking_count)
print("Duplicate seat groups:", duplicate_count)

if duplicate_count == 0:
    print("Reliability check: PASSED")
else:
    print("Reliability check: FAILED")

========== SYSTEM RELIABILITY CHECK ==========
Total buses: 4
Total bookings: 2
Duplicate seat groups: 0
Reliability check: PASSED


In [25]:
final_report = pd.DataFrame({
    "Metric": [
        "Total Buses",
        "Total Bookings",
        "Duplicate Seat Groups",
        "Available Seats for Bus 1"
    ],
    "Value": [
        bus_count,
        booking_count,
        duplicate_count,
        len(get_available_seats(1, "2026-10-01"))
    ]
})

final_report

,Metric,Value
0,Total Buses,4
1,Total Bookings,2
2,Duplicate Seat Groups,0
3,Available Seats for Bus 1,38


In [26]:
final_report.to_csv(
    "bus_pass_system_report.csv",
    index=False
)

print("Final report saved successfully!")

Final report saved successfully!


In [27]:
all_bookings = pd.read_sql_query("""
SELECT *
FROM bookings
""", connection)

all_bookings.to_csv(
    "bus_bookings.csv",
    index=False
)

print("Booking records saved successfully!")

Booking records saved successfully!


In [28]:
bus_data = pd.read_sql_query("""
SELECT *
FROM buses
""", connection)

bus_data.to_csv(
    "bus_details.csv",
    index=False
)

print("Bus details saved successfully!")

Bus details saved successfully!


In [29]:
print("""
=============================================
     CODEALPHA TASK 3
     CLOUD-BASED BUS PASS SYSTEM
=============================================

✓ Bus database created
✓ Bus search implemented
✓ Seat availability implemented
✓ Automatic seat allocation implemented
✓ Unique ticket ID generated
✓ Online-style ticket booking implemented
✓ Duplicate seat booking prevented
✓ Database-based pricing implemented
✓ Ticket verification implemented
✓ High-traffic simulation completed
✓ Reliability check completed
✓ Reports exported to CSV

=============================================
PROJECT STATUS: COMPLETED
=============================================
""")


     CODEALPHA TASK 3
     CLOUD-BASED BUS PASS SYSTEM

✓ Bus database created
✓ Bus search implemented
✓ Seat availability implemented
✓ Automatic seat allocation implemented
✓ Unique ticket ID generated
✓ Online-style ticket booking implemented
✓ Duplicate seat booking prevented
✓ Database-based pricing implemented
✓ Ticket verification implemented
✓ High-traffic simulation completed
✓ Reliability check completed
✓ Reports exported to CSV

PROJECT STATUS: COMPLETED

